In [1]:
import pandas as pd

# Load the original dataset
df = pd.read_csv(
    "../data/10millionPasswords.csv",
    dtype={"password": "string"},
    keep_default_na=False
)

print("Original number of rows:", len(df))
df.head()

Original number of rows: 999997


,rank,password
0,1,123456
1,2,password
2,3,12345678
3,4,qwerty
4,5,123456789


In [2]:
# Remove the three previously identified rows
df_clean = df.drop(index=[20917, 49135, 282774]).copy()

# Remove duplicate passwords
df_clean = df_clean.drop_duplicates(subset=["password"])

# Reset row indices
df_clean = df_clean.reset_index(drop=True)

# Display results
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))
print("Remaining duplicates:", df_clean["password"].duplicated().sum())

Original rows: 999997
Cleaned rows: 999994
Removed rows: 3
Remaining duplicates: 0


In [3]:
# Calculate password length
df_clean["length"] = df_clean["password"].str.len()

# Assign each password to a length group
df_clean["group"] = df_clean["length"].apply(
    lambda length: "short" if length <= 8 else "long"
)

# Display the number of passwords in each group
print(df_clean["group"].value_counts())

# Preview the data
df_clean.head()

group
short    816946
long     183048
Name: count, dtype: int64


,rank,password,length,group
0,1,123456,6,short
1,2,password,8,short
2,3,12345678,8,short
3,4,qwerty,6,short
4,5,123456789,9,long


In [4]:
from sklearn.model_selection import train_test_split

# Split data into training (80%) and testing (20%)
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean["group"]
)

# Display results
print("Training data:", len(train_df))
print("Testing data:", len(test_df))

print("\nTraining groups:")
print(train_df["group"].value_counts())

print("\nTesting groups:")
print(test_df["group"].value_counts())

Training data: 799995
Testing data: 199999

Training groups:
group
short    653557
long     146438
Name: count, dtype: int64

Testing groups:
group
short    163389
long      36610
Name: count, dtype: int64


In [5]:
# Separate the test data into two groups
short_test = test_df[test_df["group"] == "short"]
long_test = test_df[test_df["group"] == "long"]

# Find the size of the smaller group
sample_size = min(len(short_test), len(long_test))

# Randomly select the same number of passwords from each group
short_sample = short_test.sample(n=sample_size, random_state=42)
long_sample = long_test.sample(n=sample_size, random_state=42)

# Combine and shuffle the balanced test data
balanced_test_df = pd.concat(
    [short_sample, long_sample],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

# Display results
print("Balanced test data:", len(balanced_test_df))
print("\nGroup distribution:")
print(balanced_test_df["group"].value_counts())

Balanced test data: 73220

Group distribution:
group
long     36610
short    36610
Name: count, dtype: int64


In [6]:
from pathlib import Path

# Create the processed data folder if it doesn't exist
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save datasets
train_df.to_csv(output_dir / "train.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)
balanced_test_df.to_csv(output_dir / "balanced_test.csv", index=False)

print("Datasets saved successfully!")

Datasets saved successfully!


In [7]:
# Verify saved datasets
train_check = pd.read_csv(
    "../data/processed/train.csv",
    keep_default_na=False
)

test_check = pd.read_csv(
    "../data/processed/test.csv",
    keep_default_na=False
)

balanced_check = pd.read_csv(
    "../data/processed/balanced_test.csv",
    keep_default_na=False
)

print("Training data:", len(train_check))
print("Testing data:", len(test_check))
print("Balanced testing data:", len(balanced_check))

print("\nBalanced test groups:")
print(balanced_check["group"].value_counts())

Training data: 799995
Testing data: 199999
Balanced testing data: 73220

Balanced test groups:
group
long     36610
short    36610
Name: count, dtype: int64
